In [1]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)

import pandas as pd

import matplotlib.pyplot as plt

In [2]:
import torch

In [3]:
lsoa_id = sorted(pd.read_csv("../Output/LSOA21CD.csv")["LSOA21CD"].tolist())

embeddings = np.load('../Model/emb_all-MiniLM-L6-v2.npy')
description_topic = pd.read_csv('../Output/NLP Output/feature_extraction_origin.csv')

In [4]:
sbert_df = pd.DataFrame(embeddings)

In [5]:
import ast

description_topic['prob_vector'] = description_topic['probabilities'].apply(ast.literal_eval)

In [6]:
topic_matrix = np.vstack(description_topic['prob_vector'].values)
topic_df = pd.DataFrame(topic_matrix, columns=[f'topic_{i}' for i in range(topic_matrix.shape[1])])

In [7]:
df_all = pd.concat([description_topic, topic_df, sbert_df], axis=1)
df_nlp = df_all.drop(columns=['description', 'topic', 'probabilities', 'prob_vector'])

In [8]:
df_nlp

,LSOA21CD,year,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,...,374,375,376,377,378,379,380,381,382,383
0,E01033084,2002,2.446652e-04,2.102272e-04,1.856370e-04,1.605959e-04,2.068002e-04,9.624113e-05,8.893767e-05,1.276421e-04,...,0.065126,0.006564,0.044565,-0.049141,0.077937,0.043583,-0.106447,0.013555,-0.028803,0.000986
1,E01002676,2002,3.113932e-04,3.376181e-04,3.395465e-04,4.959566e-04,2.979200e-04,6.141201e-04,2.503583e-03,2.650973e-04,...,0.082411,0.065045,0.019736,0.052101,0.014027,0.023311,0.080643,-0.023715,0.038558,0.048565
2,E01002662,2002,2.018424e-306,2.240213e-306,2.140056e-306,9.606314e-307,1.152876e-306,6.933691e-307,5.744552e-307,6.489069e-307,...,0.014257,0.066394,-0.035941,0.029888,0.066311,0.028401,-0.005959,-0.010439,0.014559,-0.005935
3,E01002687,2002,2.227479e-01,1.119299e-03,9.470741e-04,5.334149e-04,9.124164e-04,3.766496e-04,3.579308e-04,4.531367e-04,...,0.031683,-0.035986,-0.031902,-0.043661,0.033311,0.045294,-0.027301,-0.041521,0.019800,-0.013135
4,E01002649,2002,3.546263e-04,3.606364e-04,3.005669e-04,1.606891e-04,3.023880e-04,1.098932e-04,9.685141e-05,1.338884e-04,...,0.037091,0.002158,-0.088090,-0.048079,0.015366,0.044858,-0.056914,-0.063085,0.002108,-0.064479
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113933,E01000638,2021,1.225738e-306,9.051317e-307,1.095371e-306,9.655573e-307,7.627033e-306,5.866553e-307,4.920145e-307,9.023499e-307,...,0.058522,0.015918,-0.029230,-0.057205,0.039704,0.027925,-0.019981,-0.024097,-0.020959,0.033360
113934,E01000488,2021,9.973086e-05,1.053339e-04,1.090802e-04,9.915776e-05,1.100874e-04,8.110994e-05,7.927814e-05,1.978663e-04,...,-0.011409,0.051760,0.020795,0.031972,0.061616,0.037672,0.003229,-0.034606,0.068963,0.014466
113935,E01034149,2021,1.196073e-03,1.633515e-03,1.360441e-01,7.625211e-04,1.007015e-03,5.737693e-04,4.782645e-04,6.190871e-04,...,0.055119,0.048625,0.000119,0.002588,0.073450,0.006472,-0.023461,-0.054904,-0.022584,0.032515
113936,E01000643,2021,2.030840e-04,2.164380e-04,2.427999e-04,9.816334e-04,2.267625e-04,1.281264e-03,4.618953e-04,1.785887e-04,...,0.038901,0.056045,0.031970,-0.052313,0.078986,0.006388,-0.002567,-0.048896,0.020874,0.036868


In [ ]:
from sklearn.preprocessing import normalize, StandardScaler

# === Basic checks ===
required_cols = {"LSOA21CD", "year"}
missing = required_cols - set(df_nlp.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")
df_nlp["year"] = df_nlp["year"].astype(int)

# === Identify columns ===
embedding_cols = [c for c in df_nlp.columns if str(c).isdigit()]
topic_cols = [c for c in df_nlp.columns if str(c).startswith("topic_")]
vector_cols = topic_cols + embedding_cols
if not embedding_cols:
    raise ValueError("No embedding columns (pure numeric column names) found.")

# === L2-normalize embeddings row-wise ===
df_nlp.loc[:, embedding_cols] = normalize(
    df_nlp[embedding_cols].to_numpy(dtype=np.float32, copy=True),
    norm="l2",
    axis=1
)

# === Aggregate + fill full LSOA×year grid + generate has_text ===
def aggregate_and_fill_with_mask(
    dfin: pd.DataFrame,
    years: list[int],
    lsoa_order: list[str],
    cols_to_mean: list[str],
    topic_cols: list[str],
) -> pd.DataFrame:
    """
    Returns a complete LSOA×year grid, including:
      - Mean-pooled topic/embedding values
      - has_text: whether this (LSOA, year) had original text records (1/0)
    """
    # Filter for the selected years
    dfx = dfin[dfin["year"].isin(years)].copy()

    # Count the number of text records for each (LSOA, year)
    cnt = (
        dfx.groupby(["LSOA21CD", "year"], as_index=False)
           .size()
           .rename(columns={"size": "n_texts"})
    )

    # Aggregate by mean
    if dfx.empty:
        # If no data for this period, create empty skeleton with zeros
        grid = pd.MultiIndex.from_product(
            [lsoa_order, years], names=["LSOA21CD", "year"]
        ).to_frame(index=False)
        for c in cols_to_mean:
            grid[c] = 0.0
        grid["has_text"] = 0
        return grid

    grouped = (
        dfx.groupby(["LSOA21CD", "year"], as_index=False)[cols_to_mean]
           .mean()
           .reset_index(drop=True)
    )

    # Merge with text counts to create has_text column
    grouped = grouped.merge(cnt, on=["LSOA21CD", "year"], how="left")
    grouped["has_text"] = (grouped["n_texts"] > 0).astype(np.int8)
    grouped = grouped.drop(columns=["n_texts"])

    # Build the full LSOA×year grid
    grid = pd.MultiIndex.from_product(
        [lsoa_order, sorted(years)], names=["LSOA21CD", "year"]
    ).to_frame(index=False)

    # Left-join aggregated values
    out = grid.merge(grouped, on=["LSOA21CD", "year"], how="left")

    # Fill missing numeric values with zeros; has_text with 0
    out[cols_to_mean] = out[cols_to_mean].fillna(0.0).astype(np.float32)
    out["has_text"] = out["has_text"].fillna(0).astype(np.int8)

    # Preserve the original LSOA order
    pos = {code: i for i, code in enumerate(lsoa_order)}
    out["_pos"] = out["LSOA21CD"].map(pos)
    out = (
        out.sort_values(["_pos", "year"], kind="mergesort")
           .drop(columns="_pos")
           .reset_index(drop=True)
    )
    return out

C:\Users\wbwha\AppData\Local\Temp\ipykernel_47280\653788984.py:83: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["_pos"] = out["LSOA21CD"].map(pos)
C:\Users\wbwha\AppData\Local\Temp\ipykernel_47280\653788984.py:83: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["_pos"] = out["LSOA21CD"].map(pos)


Train period shape: (49940, 1561)
Predict period shape: (49940, 1561)


In [ ]:
# === Build datasets for both periods ===
years_train   = list(range(2002, 2012))   # 2002–2011
years_predict = list(range(2012, 2022))   # 2012–2021

df_train_full = aggregate_and_fill_with_mask(df_nlp, years_train, lsoa_id, vector_cols, topic_cols)
df_predict_full = aggregate_and_fill_with_mask(df_nlp, years_predict, lsoa_id, vector_cols, topic_cols)

In [ ]:
# === Standardize topic columns ===
# Fit and transform only for rows with has_text == 1; keep zeros for has_text == 0
scaler_topic = StandardScaler(with_mean=True, with_std=True)

# Train period: fit using rows that have text
mask_tr = df_train_full["has_text"] == 1
if mask_tr.any():
    scaler_topic.fit(df_train_full.loc[mask_tr, topic_cols])
    df_train_full.loc[mask_tr, topic_cols] = scaler_topic.transform(
        df_train_full.loc[mask_tr, topic_cols]
    ).astype(np.float32)
# Rows without text remain zero

# Predict period: transform using the same scaler
mask_te = df_predict_full["has_text"] == 1
if mask_te.any():
    df_predict_full.loc[mask_te, topic_cols] = scaler_topic.transform(
        df_predict_full.loc[mask_te, topic_cols]
    ).astype(np.float32)
# Rows without text remain zero

print("Train period shape:", df_train_full.shape)
print("Predict period shape:", df_predict_full.shape)

# Save scaler for inference stage
# import joblib
# joblib.dump(scaler_topic, "topic_scaler.pkl")

In [11]:
for _df in (df_train_full, df_predict_full):
    if "has_text" not in _df.columns:
        raise ValueError("has_text column not found. Run the aggregation-with-mask step first.")
    mask = _df["has_text"] == 1
    if mask.any():
        _df.loc[mask, embedding_cols] = normalize(
            _df.loc[mask, embedding_cols].to_numpy(dtype=np.float32, copy=True),
            norm="l2",
            axis=1
        ).astype(np.float32)

In [12]:
df_train_full

,LSOA21CD,year,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,...,375,376,377,378,379,380,381,382,383,has_text
0,E01000001,2002,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0
1,E01000001,2003,-0.133201,-0.088955,-0.084737,-0.095073,-0.074624,-0.085539,-0.081094,-0.053714,...,0.001709,-0.036461,0.022549,0.086162,0.061631,-0.053037,0.027105,-0.014997,-0.043228,1
2,E01000001,2004,-0.122073,-0.077727,-0.062505,-0.077196,0.417353,-0.073517,-0.063379,-0.026341,...,-0.000922,-0.029246,-0.030955,0.013004,0.065447,0.001875,0.009545,-0.015820,0.006577,1
3,E01000001,2005,-0.129687,-0.085108,-0.075793,-0.089335,0.133206,-0.081856,-0.075776,-0.045323,...,-0.020362,-0.046676,0.004170,0.047610,0.016454,-0.009607,0.005506,-0.061554,0.029101,1
4,E01000001,2006,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49935,E01035722,2007,-0.121888,-0.062838,0.568594,-0.081824,-0.015088,-0.074152,-0.064379,-0.035631,...,-0.018530,0.023076,-0.018444,0.040244,0.011657,-0.069387,-0.074939,0.025295,0.047209,1
49936,E01035722,2008,-0.131684,-0.087076,-0.079467,-0.084941,-0.060550,-0.070725,-0.058987,-0.047305,...,0.049101,0.019952,-0.029302,0.014735,0.014762,-0.036022,-0.030350,-0.072991,0.056441,1
49937,E01035722,2009,-0.125135,-0.072628,2.037413,-0.085654,-0.040766,-0.077083,-0.064408,-0.041855,...,0.060783,-0.001028,-0.017491,0.035908,0.033462,0.035245,-0.021394,0.011778,0.001003,1
49938,E01035722,2010,-0.130170,-0.084421,-0.075909,-0.080518,-0.052108,-0.064576,0.008681,-0.042801,...,0.087707,0.002522,-0.039106,0.023317,-0.000325,0.011414,0.008109,-0.019464,0.034846,1


In [13]:
import re

def get_topk_topic_cols(topic_cols, k):
    """Return the topic column names for the first k topics (by numeric suffix)."""
    pat = re.compile(r"^topic_(\d+)$")
    indexed = []
    for c in topic_cols:
        m = pat.match(str(c))
        if m:
            indexed.append((int(m.group(1)), c))
    indexed.sort(key=lambda x: x[0])
    return [c for _, c in indexed[:k]]

def to_tensor_3d(df_period, lsoa_order, years, feature_cols, include_has_text=False):
    """
    Convert a complete LSOA×year grid DataFrame into:
      X: [N, T, D] float32 features
      M: [N, T]    bool mask (has_text)
      years_sorted: list of years (sorted)
    """
    years_sorted = sorted(years)
    N = len(lsoa_order)
    D = len(feature_cols) + (1 if include_has_text else 0)
    T = len(years_sorted)

    X = np.zeros((N, T, D), dtype=np.float32)
    M = np.zeros((N, T), dtype=bool)

    for t, y in enumerate(years_sorted):
        sub = df_period[df_period["year"] == y].set_index("LSOA21CD")

        feats = sub.loc[lsoa_order, feature_cols].to_numpy(dtype=np.float32, copy=True)
        X[:, t, :len(feature_cols)] = feats

        has_text_col = sub.loc[lsoa_order, "has_text"].to_numpy()
        M[:, t] = has_text_col.astype(bool)

        if include_has_text:
            X[:, t, -1] = has_text_col.astype(np.float32)

    return torch.from_numpy(X), torch.from_numpy(M), years_sorted

# === Build different feature sets ===
tensor_outputs = {}

# 1) Embedding only
tensor_outputs["embedding_only_train"] = to_tensor_3d(df_train_full, lsoa_id, years_train, embedding_cols)
tensor_outputs["embedding_only_pred"]  = to_tensor_3d(df_predict_full, lsoa_id, years_predict, embedding_cols)

# 2) All topics
tensor_outputs["topic_all_train"] = to_tensor_3d(df_train_full, lsoa_id, years_train, topic_cols)
tensor_outputs["topic_all_pred"]  = to_tensor_3d(df_predict_full, lsoa_id, years_predict, topic_cols)

# 3) Top-K topics
for k in [500, 200, 100, 50]:
    topk_cols = get_topk_topic_cols(topic_cols, k)
    key_train = f"topic_top{k}_train"
    key_pred  = f"topic_top{k}_pred"
    tensor_outputs[key_train] = to_tensor_3d(df_train_full, lsoa_id, years_train, topk_cols)
    tensor_outputs[key_pred]  = to_tensor_3d(df_predict_full, lsoa_id, years_predict, topk_cols)

# === Example: inspect one of them ===
for name, (X, mask, years) in tensor_outputs.items():
    print(f"{name}: X={tuple(X.shape)}, mask={tuple(mask.shape)}, years={years[:3]}...{years[-3:]}")

embedding_only_train: X=(4994, 10, 384), mask=(4994, 10), years=[2002, 2003, 2004]...[2009, 2010, 2011]
embedding_only_pred: X=(4994, 10, 384), mask=(4994, 10), years=[2012, 2013, 2014]...[2019, 2020, 2021]
topic_all_train: X=(4994, 10, 1174), mask=(4994, 10), years=[2002, 2003, 2004]...[2009, 2010, 2011]
topic_all_pred: X=(4994, 10, 1174), mask=(4994, 10), years=[2012, 2013, 2014]...[2019, 2020, 2021]
topic_top500_train: X=(4994, 10, 500), mask=(4994, 10), years=[2002, 2003, 2004]...[2009, 2010, 2011]
topic_top500_pred: X=(4994, 10, 500), mask=(4994, 10), years=[2012, 2013, 2014]...[2019, 2020, 2021]
topic_top200_train: X=(4994, 10, 200), mask=(4994, 10), years=[2002, 2003, 2004]...[2009, 2010, 2011]
topic_top200_pred: X=(4994, 10, 200), mask=(4994, 10), years=[2012, 2013, 2014]...[2019, 2020, 2021]
topic_top100_train: X=(4994, 10, 100), mask=(4994, 10), years=[2002, 2003, 2004]...[2009, 2010, 2011]
topic_top100_pred: X=(4994, 10, 100), mask=(4994, 10), years=[2012, 2013, 2014]...[201

In [15]:
import os

def save_tensor_triplet_auto(
    name: str,
    X: torch.Tensor,
    mask: torch.Tensor,
    years,                      # list[int] or 1D tensor
    save_dir: str = "../Output/Pred Input",
    save_mask: bool = True,
    save_years: bool = True
):
    """
    Save (X, mask, years) to a .pt file with standardized naming.
    - name: key from tensor_outputs, e.g. "topic_top100_train"
    - save_mask/save_years: toggle whether to include them in the file
    - add_timestamp: append YYYYMMDD_HHMMSS to the filename
    """
    os.makedirs(save_dir, exist_ok=True)

    fname = f"{name}.pt"

    path = os.path.join(save_dir, fname)
    payload = {"X": X}

    if save_mask:
        payload["mask"] = mask
    if save_years:
        payload["years"] = years

    torch.save(payload, path)
    print(f"Saved: {path}")

# === Save everything in tensor_outputs ===
# By default we save X, mask, and years for each entry.
out_dir = "../Output/Pred Input"
for name, (X, mask, years) in tensor_outputs.items():
    save_tensor_triplet_auto(
        name=name,
        X=X,
        mask=mask,
        years=years,
        save_dir=out_dir,
        save_mask=True,      # set False if you don't want to store mask
        save_years=True     # set False if you don't want to store years
    )

Saved: ../Output/Pred Input\embedding_only_train.pt
Saved: ../Output/Pred Input\embedding_only_pred.pt
Saved: ../Output/Pred Input\topic_all_train.pt
Saved: ../Output/Pred Input\topic_all_pred.pt
Saved: ../Output/Pred Input\topic_top500_train.pt
Saved: ../Output/Pred Input\topic_top500_pred.pt
Saved: ../Output/Pred Input\topic_top200_train.pt
Saved: ../Output/Pred Input\topic_top200_pred.pt
Saved: ../Output/Pred Input\topic_top100_train.pt
Saved: ../Output/Pred Input\topic_top100_pred.pt
Saved: ../Output/Pred Input\topic_top50_train.pt
Saved: ../Output/Pred Input\topic_top50_pred.pt


In [ ]:
for name, (X, mask, years) in tensor_outputs.items():
    if "train" in name:
        split = "train"
    elif "pred" in name:
        split = "predict"
    else:
        raise ValueError(f"Cannot parse split from name '{name}'")

    if name.startswith("embedding_only"):
        feature_type = "embedding"
        k = None
    elif name.startswith("topic_all"):
        feature_type = "topic"
        k = None
    elif name.startswith("topic_top"):
        import re
        match = re.search(r"topic_top(\d+)", name)
        if match:
            feature_type = "topic_topK"
            k = int(match.group(1))
        else:
            raise ValueError(f"Cannot parse k from name '{name}'")
    else:
        raise ValueError(f"Unknown feature type in '{name}'")

    save_tensor_triplet_auto(X, mask, years, split, feature_type, k)


In [17]:
Y_5 = pd.read_csv("../Output/Gentrification label_2021_5class.csv").sort_values("LSOA21CD").reset_index(drop=True)

Y_7 = pd.read_csv("../Output/Gentrification label_2021_7class.csv").sort_values("LSOA21CD").reset_index(drop=True)

In [18]:
train_not_grouped_7class = pd.merge(train_not_grouped_7class, Y_7, on='LSOA21CD', how='left')
train_not_grouped_5class = pd.merge(train_not_grouped_5class, Y_5, on='LSOA21CD', how='left')

In [19]:
label_mapping_5 = {
    "Class 0 - Stable Affluent and Moderate-income": 0,
    "Class 1 - Stable Low-income": 1,
    "Class 2 - Ongoing Displacement": 2,
    "Class 3 - At Risk of Gentrification": 3,
    "Class 4 - Ongoing Gentrification": 4
}
train_not_grouped_5class["class_int"] = train_not_grouped_5class["gentrification_class"].map(label_mapping_5)

In [20]:
label_mapping_7 = {
    "Class 0 - Stable Affluent and Moderate-income": 0,
    "Class 1 - Upgrading or Gentrified": 1,
    "Class 2 - Stable Low-income": 2,
    "Class 3 - Economically Decline": 3,
    "Class 4 - Ongoing Displacement": 4,
    "Class 5 - At Risk of Gentrification": 5,
    "Class 6 - Ongoing Gentrification": 6
}
train_not_grouped_7class["class_int"] = train_not_grouped_7class["gentrification_class"].map(label_mapping_7)

In [21]:
train_not_grouped_5class.to_csv("../Output/Pred Input/train_not_grouped_5class.csv", index=False)
train_not_grouped_7class.to_csv("../Output/Pred Input/train_not_grouped_7class.csv", index=False)

X_predict_not_grouped.to_csv("../Output/Pred Input/X_predict_not_grouped.csv", index=False)

In [22]:
def construct_temporal_tensor(
    df: pd.DataFrame,
    lsoa_id: list,
    feature_type: str = "all",  # "all", "digit", or "topic"
    id_col: str = "LSOA21CD",
    time_col: str = "year",
    train_years: list = list(range(2002, 2012)),
    pred_years: list = list(range(2012, 2022))
):
    
    # Ensure dtypes
    df[id_col] = df[id_col].astype(str)
    df[time_col] = df[time_col].astype(int)

    # Identify vector columns
    if feature_type == "sbert":
        vector_cols = [col for col in df.columns if str(col).isdigit()]
    elif feature_type == "topic":
        vector_cols = [col for col in df.columns if str(col).startswith("topic_")]
    elif feature_type == "all":
        exclude_cols = {id_col, time_col}
        vector_cols = [col for col in df.columns if col not in exclude_cols]
    else:
        raise ValueError("feature_type must be one of ['all', 'digit', 'topic']")

    # Mean-pool per (LSOA, year)
    df_grouped = (
        df.groupby([id_col, time_col])[vector_cols]
        .mean()
        .reset_index()
    )

    # Construct year and mapping
    year_list = sorted(df_grouped[time_col].unique())
    n_lsoa = len(lsoa_id)
    T = len(year_list)
    D = len(vector_cols)

    lsoa_to_idx = {lsoa: i for i, lsoa in enumerate(lsoa_id)}
    year_to_idx = {year: i for i, year in enumerate(year_list)}

    # Initialize full tensor
    X_seq = np.zeros((n_lsoa, T, D), dtype=np.float32)

    # Fill tensor
    for _, row in df_grouped.iterrows():
        lsoa = row[id_col]
        year = row[time_col]
        if lsoa in lsoa_to_idx and year in year_to_idx:
            l_idx = lsoa_to_idx[lsoa]
            y_idx = year_to_idx[year]
            X_seq[l_idx, y_idx, :] = row[vector_cols].values.astype(np.float32)

    X_seq_tensor = torch.tensor(X_seq)  # [n_lsoa, T, D]

    # Index split
    train_idx = [year_to_idx[y] for y in train_years if y in year_to_idx]
    pred_idx = [year_to_idx[y] for y in pred_years if y in year_to_idx]

    X_seq_train = X_seq_tensor[:, train_idx, :]  # [n_lsoa, T_train, D]
    X_seq_pred  = X_seq_tensor[:, pred_idx, :]   # [n_lsoa, T_pred, D]

    return X_seq_train, X_seq_pred

In [23]:
X_seq_train_sbert, X_seq_pred_sbert = construct_temporal_tensor(df=df_features_,
                                                                lsoa_id=lsoa_id,
                                                                feature_type="sbert")

In [24]:
X_seq_train_topic, X_seq_pred_topic = construct_temporal_tensor(df=df_features_,
                                                                lsoa_id=lsoa_id,
                                                                feature_type="topic")

In [ ]:
X_seq_train_all, X_seq_pred_all = construct_temporal_tensor(df=df_features_,
                                                                lsoa_id=lsoa_id,
                                                                feature_type="all")

In [25]:
torch.save(torch.tensor(X_seq_train_sbert, dtype=torch.float32), "../Output/Pred Input/X_train_lstm_sbert.pt")
torch.save(torch.tensor(X_seq_pred_sbert, dtype=torch.float32), "../Output/Pred Input/X_predict_lstm_sbert.pt")

C:\Users\wbwha\AppData\Local\Temp\ipykernel_31068\1531887212.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.save(torch.tensor(X_seq_train_sbert, dtype=torch.float32), "../Output/Pred Input/X_train_lstm_sbert.pt")
C:\Users\wbwha\AppData\Local\Temp\ipykernel_31068\1531887212.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.save(torch.tensor(X_seq_pred_sbert, dtype=torch.float32), "../Output/Pred Input/X_predict_lstm_sbert.pt")


In [26]:
torch.save(torch.tensor(X_seq_train_topic, dtype=torch.float32), "../Output/Pred Input/X_train_lstm_topic.pt")
torch.save(torch.tensor(X_seq_pred_topic, dtype=torch.float32), "../Output/Pred Input/X_predict_lstm_topic.pt")

C:\Users\wbwha\AppData\Local\Temp\ipykernel_31068\1472374149.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.save(torch.tensor(X_seq_train_topic, dtype=torch.float32), "../Output/Pred Input/X_train_lstm_topic.pt")
C:\Users\wbwha\AppData\Local\Temp\ipykernel_31068\1472374149.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.save(torch.tensor(X_seq_pred_topic, dtype=torch.float32), "../Output/Pred Input/X_predict_lstm_topic.pt")


In [ ]:
torch.save(torch.tensor(X_seq_train_all, dtype=torch.float32), "../Output/Pred Input/X_train_lstm.pt")
torch.save(torch.tensor(X_seq_pred_all, dtype=torch.float32), "../Output/Pred Input/X_predict_lstm.pt")

In [54]:
# === 1. Ensure correct dtypes and clean base
df_features_['LSOA21CD'] = df_features_['LSOA21CD'].astype(str)
df_features_['year'] = df_features_['year'].astype(int)

# === 2. Identify vector columns
vector_cols = [
    col for col in df_features_.columns
    if str(col).isdigit() or str(col).startswith("topic_")
]

# === 3. Mean-pooling per (LSOA, year)
df_grouped = (
    df_features_
    .groupby(['LSOA21CD', 'year'])[vector_cols]
    .mean()
    .reset_index()
)

# Extract year range from grouped data
year_list = sorted(df_grouped["year"].unique())
n_lsoa = len(lsoa_id)
T = len(year_list)
D = len(vector_cols)

# Index mapping
lsoa_to_idx = {lsoa: i for i, lsoa in enumerate(lsoa_id)}
year_to_idx = {year: i for i, year in enumerate(year_list)}

# Initialize tensor
X_seq = np.zeros((n_lsoa, T, D), dtype=np.float32)

# Fill values
for _, row in df_grouped.iterrows():
    lsoa = row["LSOA21CD"]
    year = row["year"]
    
    if lsoa in lsoa_to_idx and year in year_to_idx:
        l_idx = lsoa_to_idx[lsoa]
        y_idx = year_to_idx[year]
        X_seq[l_idx, y_idx, :] = row[vector_cols].values.astype(np.float32)

# Convert to tensor
X_seq_tensor = torch.tensor(X_seq)  # [n_lsoa, T, D]

# Define time periods
train_years = list(range(2002, 2012))
pred_years = list(range(2012, 2022))
year_to_idx = {year: i for i, year in enumerate(year_list)}

train_idx = [year_to_idx[y] for y in train_years if y in year_to_idx]
pred_idx = [year_to_idx[y] for y in pred_years if y in year_to_idx]

# Split tensors
X_seq_train = X_seq_tensor[:, train_idx, :]  # [n_lsoa, T_train, D]
X_seq_pred  = X_seq_tensor[:, pred_idx, :]   # [n_lsoa, T_pred, D]

In [19]:
def conversion(df_):
     
    list_of_rows = df_.values.tolist()
    array = np.array(list_of_rows) 

    df = description_topic[['LSOA21CD', 'year']].copy()
    df['vector'] = list(array)
    return df

def group_by_lsoa(year_range, df_features):
    df_sub = df_features[df_features['year'].isin(year_range)]
    grouped = df_sub.groupby('LSOA21CD')['vector'].apply(
        lambda v: np.mean(np.stack(v), axis=0)
    )
    return grouped.sort_index()

def fill_missing(vecs, lsoa_list, dim):
    X = np.zeros((len(lsoa_list), dim))
    for i, lsoa in enumerate(lsoa_list):
        if lsoa in vecs:
            X[i] = vecs[lsoa]
    return X

In [20]:
X_standard

array([[ 0.89221555,  0.65585244,  0.28538004, ..., -0.06126561,
        -0.04123121, -0.05699108],
       [-0.61536992,  1.18830884,  1.66928613, ..., -0.011382  ,
        -0.04225199,  0.02623823],
       [ 1.67322409, -0.39132032,  0.22761244, ..., -0.07107525,
        -0.07535197, -0.06595932],
       ...,
       [ 1.85581326,  1.5247457 , -0.21956787, ..., -0.01352985,
         0.05026182, -0.0146278 ],
       [-0.45884705, -0.85070032,  0.02964709, ...,  0.04602844,
        -0.05261898,  0.00941492],
       [ 0.7641021 ,  0.76497895,  1.47620475, ..., -0.07107525,
        -0.07535197, -0.06595932]])

In [ ]:
topic_df_50 = topic_df.iloc[:, :50]
topic_df_100 = topic_df.iloc[:, :100]
topic_df_200 = topic_df.iloc[:, :200]
topic_df_500 = topic_df.iloc[:, :500]

In [ ]:
df_topic_50 = conversion(topic_df_50)
df_topic_100 = conversion(topic_df_100)
df_topic_200 = conversion(topic_df_200)
df_topic_500 = conversion(topic_df_500)

In [ ]:
df_topic = conversion(topic_df)
df_sbert = conversion(sbert_df)

In [ ]:
train_vecs_topic_50 = group_by_lsoa(range(2002, 2012), df_topic_50)
train_vecs_topic_100 = group_by_lsoa(range(2002, 2012), df_topic_100)
train_vecs_topic_200 = group_by_lsoa(range(2002, 2012), df_topic_200)
train_vecs_topic_500 = group_by_lsoa(range(2002, 2012), df_topic_500)

In [ ]:
train_vecs_topic = group_by_lsoa(range(2002, 2012), topic_scaled_df)
train_vecs_sbert = group_by_lsoa(range(2002, 2012), sbert_scaled_df)

In [ ]:
train_vecs = group_by_lsoa(range(2002, 2012))
predict_vecs = group_by_lsoa(range(2012, 2022))

In [ ]:
X_train_topic_50 = fill_missing(train_vecs_topic_50, lsoa_id, dim=50)
X_train_topic_100 = fill_missing(train_vecs_topic_100, lsoa_id, dim=100)
X_train_topic_200 = fill_missing(train_vecs_topic_200, lsoa_id, dim=200)
X_train_topic_500 = fill_missing(train_vecs_topic_500, lsoa_id, dim=500)

In [ ]:
X_train_topic = fill_missing(train_vecs_topic, lsoa_id, dim=1174)
X_train_sbert = fill_missing(train_vecs_sbert, lsoa_id, dim=384)

In [ ]:
X_train = fill_missing(train_vecs, lsoa_id, feature_dim)
X_predict = fill_missing(predict_vecs, lsoa_id, feature_dim)

In [37]:
label_mapping_2 = {
    "Class 0 - Affluent and Middle Class": 0,
    "Class 1 - Stable Low-income": 0,
    "Class 2 - Ongoing Displacement": 0,
    "Class 3 - At Risk of Gentrification": 0,
    "Class 4 - Ongoing Gentrification": 1
}

Y_5["label_int_"] = Y_5["gentrification_class"].map(label_mapping_2)

label_dict_2 = dict(zip(Y_5["LSOA21CD"], Y_5["label_int_"]))

y_train_2 = torch.full((len(lsoa_id),), -1, dtype=torch.long)

for i, lsoa in enumerate(lsoa_id):
    if lsoa in label_dict_2:
        y_train_2[i] = label_dict_2[lsoa]

In [57]:
label_mapping_5 = {
    "Class 0 - Affluent and Middle Class": 0,
    "Class 1 - Stable Low-income": 1,
    "Class 2 - Ongoing Displacement": 2,
    "Class 3 - At Risk of Gentrification": 3,
    "Class 4 - Ongoing Gentrification": 4
}

Y_5["label_int"] = Y_5["gentrification_class"].map(label_mapping_5)

label_dict_5 = dict(zip(Y_5["LSOA21CD"], Y_5["label_int"]))

y_train_5 = torch.full((len(lsoa_id),), -1, dtype=torch.long)

for i, lsoa in enumerate(lsoa_id):
    if lsoa in label_dict_5:
        y_train_5[i] = label_dict_5[lsoa]

In [59]:
label_mapping_7 = {
    "Class 0 - Stable Affluent and Moderate-income": 0,
    "Class 1 - Upgrading or Gentrified": 1,
    "Class 2 - Stable Low-income": 2,
    "Class 3 - Economically Decline": 3,
    "Class 4 - Ongoing Displacement": 4,
    "Class 5 - At Risk of Gentrification": 5,
    "Class 6 - Ongoing Gentrification": 6
}

Y_7["label_int"] = Y_7["gentrification_class"].map(label_mapping_7)

label_dict_7 = dict(zip(Y_7["LSOA21CD"], Y_7["label_int"]))

y_train_7 = torch.full((len(lsoa_id),), -1, dtype=torch.long)

for i, lsoa in enumerate(lsoa_id):
    if lsoa in label_dict_7:
        y_train_7[i] = label_dict_7[lsoa]

In [ ]:
torch.save(torch.tensor(X_train_topic_50, dtype=torch.float32), "../Output/Pred Input/X_train_topic_50.pt")
torch.save(torch.tensor(X_train_topic_100, dtype=torch.float32), "../Output/Pred Input/X_train_topic_100.pt")
torch.save(torch.tensor(X_train_topic_200, dtype=torch.float32), "../Output/Pred Input/X_train_topic_200.pt")
torch.save(torch.tensor(X_train_topic_200, dtype=torch.float32), "../Output/Pred Input/X_train_topic_200.pt")

In [ ]:
torch.save(torch.tensor(X_train_topic, dtype=torch.float32), "../Output/Pred Input/X_train_topic.pt")
torch.save(torch.tensor(X_train_sbert, dtype=torch.float32), "../Output/Pred Input/X_train_sbert.pt")

In [ ]:
torch.save(torch.tensor(X_train, dtype=torch.float32), "../Output/Pred Input/X_train.pt")
torch.save(torch.tensor(X_predict, dtype=torch.float32), "../Output/Pred Input/X_predict.pt")

In [ ]:
torch.save(y_train_2, "../Output/Pred Input/y_train_2.pt")
torch.save(y_train_5, "../Output/Pred Input/y_train_5.pt")
torch.save(y_train_7, "../Output/Pred Input/y_train_7.pt")